# Optuna — búsqueda de hiperparámetros para soft-clDice (nnU-Net)

## Experimento 13 — ImageCAS

**Objetivo:** buscar de forma automática el mejor peso para cada uno de los tres términos de la pérdida combinada, en lugar de barrer valores manualmente (como en los notebooks 12, 12b y 18):

- `weight_dice`: peso del término Dice (dentro de `DC_and_CE_loss`, nativo de nnU-Net, fijado previamente en 1).
- `weight_ce`: peso del término de entropía cruzada (ídem).
- `lambda_cldice`: peso del término topológico (clDice).

Se incluye además un cuarto hiperparámetro de carácter estructural, que no es un peso de combinación sino un parámetro de la propia operación de skeletonización:
- `skeleton_iterations`: número de iteraciones de soft-skeletonization.

**Optimización multiobjetivo** (Optuna, sin escalarizar a un único valor): Dice ↑, clDice ↑, número de componentes conexas ↓ y HD95 crudo ↓. El resultado es un **frente de Pareto** y no un único óptimo; la mejor combinación depende de la prioridad asignada (mayor conectividad frente a mayor Dice).

**Coste:** cada trial entrena con `nnUNetTrainer_50epochs` (en lugar de 100 épocas) para que la búsqueda sea viable. Con ≈ 50 s/época según los logs previos (A100), cada trial requiere entre 45 y 55 min (entrenamiento y validación de 40 casos). `N_TRIALS` se ajusta según el tiempo de GPU disponible.

**Requisitos previos:** `nnUNet_raw` y `nnUNet_preprocessed` de `Dataset501_ImageCAS` montados en Drive (igual que en los notebooks 11, 12 y 12b). Este notebook no repite el preprocesado y reutiliza los datos existentes.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")

PyTorch: 2.11.0+cu128
CUDA disponible: True
GPU: NVIDIA A100-SXM4-80GB
VRAM: 85.09 GB


In [2]:
!python -m pip install -q nnunetv2 optuna medpy scikit-image

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 291.1/291.1 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.8/102.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 91.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
!rm -rf /content/drive

In [2]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [3]:
import os
from pathlib import Path

BASE = Path("/content/drive/MyDrive/TFM_nnUNet")
NNUNET_RAW = BASE / "nnUNet_raw"
NNUNET_PREPROCESSED = BASE / "nnUNet_preprocessed"
NNUNET_RESULTS = BASE / "nnUNet_results"
CUSTOM_TRAINERS = BASE / "custom_trainers"
CUSTOM_TRAINERS.mkdir(parents=True, exist_ok=True)

os.environ["nnUNet_raw"] = str(NNUNET_RAW)
os.environ["nnUNet_preprocessed"] = str(NNUNET_PREPROCESSED)
os.environ["nnUNet_results"] = str(NNUNET_RESULTS)
os.environ["nnUNet_extTrainer"] = str(CUSTOM_TRAINERS)

DATASET_ID = 501
DATASET_FOLDER_NAME = "Dataset501_ImageCAS"
CONFIGURATION = "3d_fullres"
FOLD = 0

EPOCHS_POR_TRIAL = 50       # nnUNetTrainer_50epochs ya existe de fabrica
TOTAL_TRIALS_DESEADOS = 15    # objetivo final del estudio completo
N_TRIALS_POR_SESION = 3       # cuantos trials lanzar cada vez que abres el notebook
                               # (≈ 50 min/trial; 3 trials ≈ 2.5 h, dentro de una sesión de Colab)
MIN_VOXELS_FILTRO = 20        # mismo umbral que en el notebook 19

GT_FOLDER = NNUNET_PREPROCESSED / DATASET_FOLDER_NAME / "gt_segmentations"

STUDY_NAME = "cldice_hparam_search"
STORAGE_PATH = f"sqlite:///{BASE / 'optuna_cldice_study.db'}"
CSV_RESULTADOS = BASE / "optuna_cldice_trials.csv"

print("nnUNet_extTrainer:", os.environ["nnUNet_extTrainer"])
print("GT_FOLDER existe:", GT_FOLDER.exists())

nnUNet_extTrainer: /content/drive/MyDrive/TFM_nnUNet/custom_trainers
GT_FOLDER existe: True


## 1. Funciones de métricas

Idénticas a las del notebook 19 (`dice_score`, `hard_cldice` con `skimage.skeletonize`, componentes conexas con conectividad 26 y HD95 mediante `medpy`), de modo que los resultados de la búsqueda sean directamente comparables con las tablas de baseline, v1, v2 y barrido de λ.


In [4]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
from scipy import ndimage
from skimage.morphology import skeletonize
from medpy.metric.binary import hd95 as medpy_hd95

CONECTIVIDAD_26 = np.ones((3, 3, 3), dtype=int)


def cargar_volumen(path):
    img = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(img).astype(np.uint8)
    spacing = img.GetSpacing()[::-1]
    return array, spacing


def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    return (2.0 * inter) / (pred.sum() + gt.sum() + 1e-8)


def hard_cldice(pred, gt):
    skel_pred = skeletonize(pred.astype(bool))
    skel_gt = skeletonize(gt.astype(bool))
    tprec = (skel_pred & gt).sum() / (skel_pred.sum() + 1e-8)
    tsens = (skel_gt & pred).sum() / (skel_gt.sum() + 1e-8)
    return (2 * tprec * tsens) / (tprec + tsens + 1e-8)


def componentes_conexas(pred):
    labeled, n = ndimage.label(pred, structure=CONECTIVIDAD_26)
    return labeled, n


def cobertura_componente_principal(pred, gt):
    labeled, n = componentes_conexas(pred)
    if n == 0:
        return 0.0, 0
    tamanos = ndimage.sum(pred, labeled, index=range(1, n + 1))
    idx_mayor = np.argmax(tamanos) + 1
    componente_mayor = labeled == idx_mayor
    cobertura = (componente_mayor & gt.astype(bool)).sum() / (gt.sum() + 1e-8)
    return cobertura, n


def hd95_seguro(pred, gt, spacing):
    if pred.sum() == 0 or gt.sum() == 0:
        return np.nan
    return medpy_hd95(pred, gt, voxelspacing=spacing)


def calcular_metricas_caso(pred_path, gt_path):
    pred, spacing = cargar_volumen(pred_path)
    gt, _ = cargar_volumen(gt_path)

    dice = dice_score(pred, gt)
    cldice = hard_cldice(pred, gt)
    cobertura, n_componentes = cobertura_componente_principal(pred, gt)
    hd95_crudo = hd95_seguro(pred, gt, spacing)

    return {
        "dice": dice,
        "cldice": cldice,
        "n_componentes": n_componentes,
        "cobertura_principal": cobertura,
        "hd95_crudo_mm": hd95_crudo,
    }


def evaluar_trainer(trainer_name):
    carpeta_val = (
        NNUNET_RESULTS / DATASET_FOLDER_NAME
        / f"{trainer_name}__nnUNetPlans__{CONFIGURATION}"
        / f"fold_{FOLD}" / "validation"
    )
    preds = sorted(carpeta_val.glob("*.nii.gz"))
    if not preds:
        raise RuntimeError(f"No se encontraron predicciones en {carpeta_val}")

    filas = []
    for pred_path in preds:
        gt_path = GT_FOLDER / pred_path.name
        if not gt_path.exists():
            continue
        filas.append(calcular_metricas_caso(pred_path, gt_path))

    df = pd.DataFrame(filas)
    return {
        "dice_mean": df["dice"].mean(),
        "cldice_mean": df["cldice"].mean(),
        "n_componentes_mean": df["n_componentes"].mean(),
        "hd95_crudo_mean": df["hd95_crudo_mm"].mean(skipna=True),
        "n_casos": len(df),
    }

print("Funciones de metricas cargadas.")

Funciones de metricas cargadas.


## 2. Verificación con caso sintético

Antes de lanzar trials de varias horas, se comprueba que las funciones devuelven resultados correctos sobre un tubo cilíndrico sintético. Como en el notebook 19, se utiliza un tubo y no un cubo sólido, ya que sobre un cubo el esqueleto de `skimage.skeletonize` degenera a un conjunto vacío.


In [5]:
zz, yy, xx = np.mgrid[0:40, 0:15, 0:15]
tubo = (((yy - 7) ** 2 + (xx - 7) ** 2) <= 4 ** 2).astype(np.uint8)

print("dice:", dice_score(tubo, tubo))
print("cldice:", hard_cldice(tubo, tubo))
_, n = componentes_conexas(tubo)
print("n_componentes:", n)
print("OK: deberian salir dice=1.0, cldice=1.0, n_componentes=1")

dice: 0.999999999997449
cldice: 0.9999999947222222
n_componentes: 1
OK: deberian salir dice=1.0, cldice=1.0, n_componentes=1


## 3. Plantilla del trainer (parametrizada por trial)

Se asigna un peso independiente a cada uno de los tres términos de la pérdida:
- `weight_dice` y `weight_ce`: pesos nativos de `DC_and_CE_loss` de nnU-Net, fijados previamente en 1 y expuestos ahora como hiperparámetros.
- `lambda_cldice`: peso del término topológico (clDice).

`skeleton_iterations` se mantiene como cuarto hiperparámetro. Es de carácter estructural (número de iteraciones de skeletonización) y no un peso de combinación de términos.


In [6]:
TRAINER_TEMPLATE = r'''
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from nnunetv2.training.loss.compound_losses import DC_and_CE_loss
from nnunetv2.training.loss.deep_supervision import DeepSupervisionWrapper
from nnunetv2.training.loss.dice import MemoryEfficientSoftDiceLoss

from nnunetv2.training.nnUNetTrainer.variants.training_length.nnUNetTrainer_Xepochs import (
    nnUNetTrainer_{epochs}epochs
)


def soft_erode_3d(x):
    p1 = -F.max_pool3d(-x, kernel_size=(3, 1, 1), stride=1, padding=(1, 0, 0))
    p2 = -F.max_pool3d(-x, kernel_size=(1, 3, 1), stride=1, padding=(0, 1, 0))
    p3 = -F.max_pool3d(-x, kernel_size=(1, 1, 3), stride=1, padding=(0, 0, 1))
    return torch.minimum(torch.minimum(p1, p2), p3)


def soft_dilate_3d(x):
    return F.max_pool3d(x, kernel_size=3, stride=1, padding=1)


def soft_open_3d(x):
    return soft_dilate_3d(soft_erode_3d(x))


def soft_skeletonize_3d(x, iterations=10):
    opened = soft_open_3d(x)
    skeleton = F.relu(x - opened)
    for _ in range(iterations):
        x = soft_erode_3d(x)
        opened = soft_open_3d(x)
        delta = F.relu(x - opened)
        skeleton = skeleton + F.relu(delta - skeleton * delta)
    return skeleton


class SoftClDiceLoss(nn.Module):
    def __init__(self, iterations=10, smooth=1e-5):
        super().__init__()
        self.iterations = iterations
        self.smooth = smooth

    def forward(self, prediction, target):
        prediction = prediction.float()
        target = target.float()
        skel_pred = soft_skeletonize_3d(prediction, iterations=self.iterations)
        skel_true = soft_skeletonize_3d(target, iterations=self.iterations)
        tprec = ((skel_pred * target).sum() + self.smooth) / (skel_pred.sum() + self.smooth)
        tsens = ((skel_true * prediction).sum() + self.smooth) / (skel_true.sum() + self.smooth)
        cldice = (2.0 * tprec * tsens) / (tprec + tsens + self.smooth)
        return 1.0 - cldice


class DiceCEPlusSoftClDice(nn.Module):
    """v1 (clDice en todas las escalas de deep supervision). regional_loss_ds
    ya lleva weight_dice/weight_ce aplicados internamente (via DC_and_CE_loss),
    asi que aqui solo se pondera el termino topologico con lambda_cldice."""

    def __init__(self, regional_loss_ds, cldice_loss, lambda_cldice):
        super().__init__()
        self.regional_loss_ds = regional_loss_ds
        self.cldice_loss = cldice_loss
        self.lambda_cldice = lambda_cldice

    def forward(self, outputs, targets):
        loss_regional = self.regional_loss_ds(outputs, targets)

        if isinstance(outputs, (list, tuple)):
            logits = outputs[0]
            target = targets[0]
        else:
            logits = outputs
            target = targets

        probabilities = torch.softmax(logits.float(), dim=1)
        foreground_probability = probabilities[:, 1:2]

        if target.ndim == logits.ndim - 1:
            target = target.unsqueeze(1)
        foreground_target = (target == 1).float()

        topology_loss = self.cldice_loss(foreground_probability, foreground_target)

        return loss_regional + self.lambda_cldice * topology_loss


class {class_name}(nnUNetTrainer_{epochs}epochs):

    def __init__(self, plans, configuration, fold, dataset_json, device=torch.device("cuda")):
        super().__init__(plans, configuration, fold, dataset_json, device)
        self.weight_dice = {weight_dice}
        self.weight_ce = {weight_ce}
        self.lambda_cldice = {lambda_cldice}
        self.skeleton_iterations = {skeleton_iterations}

    def on_train_start(self):
        super().on_train_start()
        self.print_to_log_file(
            f"[Optuna trial] weight_dice={{self.weight_dice:.4f}} "
            f"weight_ce={{self.weight_ce:.4f}} "
            f"lambda_cldice={{self.lambda_cldice:.4f}} "
            f"skeleton_iterations={{self.skeleton_iterations}}"
        )

    def _build_loss(self):
        regional_loss = DC_and_CE_loss(
            soft_dice_kwargs={{
                "batch_dice": self.configuration_manager.batch_dice,
                "smooth": 1e-5,
                "do_bg": False,
                "ddp": self.is_ddp
            }},
            ce_kwargs={{}},
            weight_ce=self.weight_ce,
            weight_dice=self.weight_dice,
            ignore_label=self.label_manager.ignore_label,
            dice_class=MemoryEfficientSoftDiceLoss
        )

        if self.enable_deep_supervision:
            deep_supervision_scales = self._get_deep_supervision_scales()
            weights = np.array([1 / (2 ** i) for i in range(len(deep_supervision_scales))])
            weights[-1] = 0
            weights = weights / weights.sum()
            regional_loss_ds = DeepSupervisionWrapper(regional_loss, weights)
        else:
            regional_loss_ds = regional_loss

        cldice_loss = SoftClDiceLoss(iterations=self.skeleton_iterations, smooth=1e-5)

        return DiceCEPlusSoftClDice(
            regional_loss_ds=regional_loss_ds,
            cldice_loss=cldice_loss,
            lambda_cldice=self.lambda_cldice,
        )
'''


def escribir_trainer(class_name, epochs, weight_dice, weight_ce, lambda_cldice, skeleton_iterations):
    codigo = TRAINER_TEMPLATE.format(
        class_name=class_name,
        epochs=epochs,
        weight_dice=weight_dice,
        weight_ce=weight_ce,
        lambda_cldice=lambda_cldice,
        skeleton_iterations=skeleton_iterations,
    )
    ruta = CUSTOM_TRAINERS / f"{class_name}.py"
    ruta.write_text(codigo)
    return ruta

print("Plantilla de trainer lista.")

Plantilla de trainer lista.


## 4. Función objetivo de Optuna

Cada trial genera el trainer con una combinación de hiperparámetros, lo entrena (`EPOCHS_POR_TRIAL` épocas), lo evalúa sobre los 40 casos de validación con las métricas del notebook 19 y guarda el resultado en un CSV de forma incremental, para conservar los datos ante una posible desconexión de Colab.


In [7]:
import subprocess
import shutil
import optuna


def objective(trial: optuna.Trial):
    weight_dice = trial.suggest_float("weight_dice", 0.3, 2.0)
    weight_ce = trial.suggest_float("weight_ce", 0.3, 2.0)
    lambda_cldice = trial.suggest_float("lambda_cldice", 0.05, 0.6)
    skeleton_iterations = trial.suggest_int("skeleton_iterations", 3, 15)

    class_name = f"nnUNetTrainerOptuna_trial{trial.number}_{EPOCHS_POR_TRIAL}epochs"

    escribir_trainer(
        class_name=class_name,
        epochs=EPOCHS_POR_TRIAL,
        weight_dice=weight_dice,
        weight_ce=weight_ce,
        lambda_cldice=lambda_cldice,
        skeleton_iterations=skeleton_iterations,
    )

    comando = [
        "nnUNetv2_train", str(DATASET_ID), CONFIGURATION, str(FOLD),
        "-tr", class_name, "--npz",
    ]

    print(f"\n=== Trial {trial.number}: weight_dice={weight_dice:.3f} "
          f"weight_ce={weight_ce:.3f} lambda_cldice={lambda_cldice:.3f} "
          f"iters={skeleton_iterations} ===")

    resultado = subprocess.run(comando, capture_output=True, text=True)
    if resultado.returncode != 0:
        print(resultado.stdout[-2000:])
        print(resultado.stderr[-2000:])
        raise RuntimeError(f"Entrenamiento del trial {trial.number} fallo")

    metricas = evaluar_trainer(class_name)

    fila = {
        "trial": trial.number,
        "weight_dice": weight_dice,
        "weight_ce": weight_ce,
        "lambda_cldice": lambda_cldice,
        "skeleton_iterations": skeleton_iterations,
        **metricas,
    }
    pd.DataFrame([fila]).to_csv(
        CSV_RESULTADOS, mode="a",
        header=not CSV_RESULTADOS.exists(), index=False,
    )

        # Opcional: eliminación de los checkpoints del trial para ahorrar espacio en Drive
    # carpeta_modelo = (NNUNET_RESULTS / DATASET_FOLDER_NAME
    #                    / f"{class_name}__nnUNetPlans__{CONFIGURATION}")
    # shutil.rmtree(carpeta_modelo, ignore_errors=True)

    return (
        metricas["dice_mean"],
        metricas["cldice_mean"],
        metricas["n_componentes_mean"],
        metricas["hd95_crudo_mean"],
    )

print("Funcion objetivo lista.")

Funcion objetivo lista.


## 5. Previsualización del espacio de búsqueda (sin entrenamiento)

Generación de N combinaciones de `weight_dice`, `weight_ce`, `lambda_cldice` y `skeleton_iterations` tal como las muestrearía Optuna, sin lanzar ningún entrenamiento. Permite comprobar que los rangos definidos en `objective()` son adecuados antes de invertir horas de GPU.


In [8]:
# Estudio auxiliar, solo para previsualizar el espacio de búsqueda
estudio_preview = optuna.create_study(directions=["maximize", "maximize", "minimize", "minimize"])

filas_preview = []
for _ in range(30):
    trial = estudio_preview.ask()
    weight_dice = trial.suggest_float("weight_dice", 0.3, 2.0)
    weight_ce = trial.suggest_float("weight_ce", 0.3, 2.0)
    lambda_cldice = trial.suggest_float("lambda_cldice", 0.05, 0.6)
    skeleton_iterations = trial.suggest_int("skeleton_iterations", 3, 15)

    filas_preview.append({
        "weight_dice": round(weight_dice, 3),
        "weight_ce": round(weight_ce, 3),
        "lambda_cldice": round(lambda_cldice, 3),
        "skeleton_iterations": skeleton_iterations,
    })
        # No se llama a estudio_preview.tell(): el trial no queda registrado como
        # real y no interfiere con el estudio definitivo (que usa su propio
        # storage en SQLite, definido en la celda siguiente)

df_preview = pd.DataFrame(filas_preview)
print(df_preview.to_string(index=False))
print()
print("Rango weight_dice:", df_preview["weight_dice"].min(), "-", df_preview["weight_dice"].max())
print("Rango weight_ce:", df_preview["weight_ce"].min(), "-", df_preview["weight_ce"].max())
print("Rango lambda_cldice:", df_preview["lambda_cldice"].min(), "-", df_preview["lambda_cldice"].max())
print("Rango skeleton_iterations:", df_preview["skeleton_iterations"].min(), "-", df_preview["skeleton_iterations"].max())

[I 2026-09-12 07:54:43,311] A new study created in memory with name: no-name-17f632e5-d1a5-4073-a819-460cfc9ac0f7


 weight_dice  weight_ce  lambda_cldice  skeleton_iterations
       0.823      1.180          0.444                   11
       1.826      0.834          0.562                   14
       1.391      0.313          0.056                    5
       1.260      0.320          0.471                    8
       1.628      0.947          0.130                    4
       1.365      0.835          0.598                    7
       1.299      1.997          0.340                    5
       0.690      1.467          0.078                    9
       1.786      1.612          0.084                   11
       0.520      0.745          0.208                   12
       1.472      0.438          0.174                    6
       1.352      1.488          0.065                    5
       1.707      0.894          0.350                    7
       1.464      1.910          0.138                   12
       1.033      1.017          0.280                   13
       0.805      0.656          0.152  

## 6. Ejecución del estudio

Estudio multiobjetivo con `storage` en SQLite en Drive, **reanudable entre trials completos**: ante una desconexión de Colab entre dos trials, basta con volver a ejecutar esta celda (y la anterior, si es necesario reconectar el runtime) para continuar desde el punto de interrupción.

Si la desconexión ocurre **durante** un trial (mientras `nnUNetv2_train` está en ejecución), dicho trial se pierde: no queda registrado en el estudio y su carpeta de checkpoints parciales en `nnUNet_results` queda huérfana. La celda siguiente muestra el número de trials completados antes de lanzar más, y esta celda solicita únicamente `N_TRIALS_POR_SESION` en cada ejecución (no el total), de modo que cada ejecución cabe en una sesión de Colab.


In [9]:
# Número de trials completados en el estudio guardado
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE_PATH,
    load_if_exists=True,
    directions=["maximize", "maximize", "minimize", "minimize"],
)

completados = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
print(f"Trials completados hasta ahora: {completados} / {TOTAL_TRIALS_DESEADOS}")

trials_restantes = max(0, TOTAL_TRIALS_DESEADOS - completados)
trials_esta_sesion = min(N_TRIALS_POR_SESION, trials_restantes)
print(f"Trials a lanzar en esta sesion: {trials_esta_sesion}")

if trials_restantes == 0:
    print("Se alcanzó el total de trials previsto. El frente de Pareto se revisa en la celda siguiente.")

[I 2026-09-12 07:54:45,566] Using an existing study with `study_name='cldice_hparam_search'` instead of creating a new one.


Trials completados hasta ahora: 15 / 15
Trials a lanzar en esta sesion: 0
Ya tienes el total de trials deseado. Revisa el frente de Pareto en la celda siguiente.


In [10]:
if trials_esta_sesion > 0:
    study.optimize(objective, n_trials=trials_esta_sesion)

print("\n=== Frente de Pareto (mejores compromisos encontrados hasta ahora) ===")
for t in study.best_trials:
    print(
        f"Trial {t.number}: params={t.params} "
        f"-> Dice={t.values[0]:.4f}, clDice={t.values[1]:.4f}, "
        f"n_comp={t.values[2]:.2f}, HD95={t.values[3]:.2f}mm"
    )

print(f"\nResultados completos: {CSV_RESULTADOS}")
print(f"Estudio (reanudable): {STORAGE_PATH}")


=== Frente de Pareto (mejores compromisos encontrados hasta ahora) ===
Trial 4: params={'weight_dice': 0.7710775680939648, 'weight_ce': 0.8753405639536114, 'lambda_cldice': 0.08452581496721129, 'skeleton_iterations': 4} -> Dice=0.7338, clDice=0.8032, n_comp=21.02, HD95=25.08mm

Resultados completos: /content/drive/MyDrive/TFM_nnUNet/optuna_cldice_trials.csv
Estudio (reanudable): sqlite:////content/drive/MyDrive/TFM_nnUNet/optuna_cldice_study.db


In [11]:
print("test")

test


## 7. Limpieza opcional de carpetas huérfanas de trials interrumpidos

Una sesión interrumpida durante un trial deja en `nnUNet_results` una carpeta con checkpoints parciales que Optuna no registra (no aparece en `study.trials`). La celda detecta estas carpetas comparando las `nnUNetTrainerOptuna_trial*` presentes en disco con los números de trial del estudio, y permite decidir su eliminación.


In [12]:
import re

carpetas_en_disco = sorted(NNUNET_RESULTS.glob(f"{DATASET_FOLDER_NAME}/nnUNetTrainerOptuna_trial*"))
trials_registrados = {t.number for t in study.trials}

huerfanas = []
for carpeta in carpetas_en_disco:
    m = re.search(r"trial(\d+)_", carpeta.name)
    if m and int(m.group(1)) not in trials_registrados:
        huerfanas.append(carpeta)

if not huerfanas:
    print("No hay carpetas huerfanas.")
else:
    print("Carpetas huerfanas encontradas (trial interrumpido, sin registrar en Optuna):")
    for c in huerfanas:
        print(" -", c)
    print("\nDescomenta las lineas de abajo para borrarlas y liberar espacio en Drive:")
    # import shutil
    # for c in huerfanas:
    #     shutil.rmtree(c, ignore_errors=True)
    # print("Borradas.")

No hay carpetas huerfanas.


## 8. Siguiente paso

Una vez identificado el frente de Pareto, se selecciona la combinación que mejor equilibre las prioridades del experimento (mayor Dice frente a mayor conectividad) y se **reentrena con 100 épocas completas** (como baseline, v1 y v2), reutilizando `escribir_trainer(..., epochs=100, ...)` de la sección 3. Los valores de este notebook corresponden a la fase de búsqueda (50 épocas) y no constituyen el resultado definitivo.
